In [1]:
from XeroGraph import xa
import pandas as pd
import numpy as np
import json
import io
import contextlib

In [2]:
# Načtení datasetu
df = pd.read_csv("../data/AirQualityUCI.csv", sep=';', decimal=',')

# Odstranění prázdných sloupců
df = df.loc[:, ~df.columns.str.contains('^Unnamed')]

# Odstranění prázdných řádků
df.dropna(how='all', inplace=True)

df = df.replace(-200, np.nan)

df.head()

,Date,Time,CO(GT),PT08.S1(CO),NMHC(GT),C6H6(GT),PT08.S2(NMHC),NOx(GT),PT08.S3(NOx),NO2(GT),PT08.S4(NO2),PT08.S5(O3),T,RH,AH
0,10/03/2004,18.00.00,2.6,1360.0,150.0,11.9,1046.0,166.0,1056.0,113.0,1692.0,1268.0,13.6,48.9,0.7578
1,10/03/2004,19.00.00,2.0,1292.0,112.0,9.4,955.0,103.0,1174.0,92.0,1559.0,972.0,13.3,47.7,0.7255
2,10/03/2004,20.00.00,2.2,1402.0,88.0,9.0,939.0,131.0,1140.0,114.0,1555.0,1074.0,11.9,54.0,0.7502
3,10/03/2004,21.00.00,2.2,1376.0,80.0,9.2,948.0,172.0,1092.0,122.0,1584.0,1203.0,11.0,60.0,0.7867
4,10/03/2004,22.00.00,1.6,1272.0,51.0,6.5,836.0,131.0,1205.0,116.0,1490.0,1110.0,11.2,59.6,0.7888


In [3]:
# Výběr pouze numerických sloupců
df_numeric = df.select_dtypes(include=[np.number])
df_numeric.head()

,CO(GT),PT08.S1(CO),NMHC(GT),C6H6(GT),PT08.S2(NMHC),NOx(GT),PT08.S3(NOx),NO2(GT),PT08.S4(NO2),PT08.S5(O3),T,RH,AH
0,2.6,1360.0,150.0,11.9,1046.0,166.0,1056.0,113.0,1692.0,1268.0,13.6,48.9,0.7578
1,2.0,1292.0,112.0,9.4,955.0,103.0,1174.0,92.0,1559.0,972.0,13.3,47.7,0.7255
2,2.2,1402.0,88.0,9.0,939.0,131.0,1140.0,114.0,1555.0,1074.0,11.9,54.0,0.7502
3,2.2,1376.0,80.0,9.2,948.0,172.0,1092.0,122.0,1584.0,1203.0,11.0,60.0,0.7867
4,1.6,1272.0,51.0,6.5,836.0,131.0,1205.0,116.0,1490.0,1110.0,11.2,59.6,0.7888


In [4]:
# Inicializace testovacího objektu
xg_test = xa(df_numeric, save_files=True, save_path="../missing/export_mechanisms")

In [5]:
# Spuštění MCAR testu (Little’s test)
xg_test.mcar()

Little's MCAR Test Results:
  Chi-square  = 5478.1911
  df          = 88
  p-value     = 0.0000
  # of patterns = 14
Missing Data Summary:
                       CO(GT)  PT08.S1(CO)     NMHC(GT)    C6H6(GT)  \
Number Missing   1683.000000   366.000000  8443.000000  366.000000   
Percent Missing     0.179865     0.039115     0.902319    0.039115   

                 PT08.S2(NMHC)      NOx(GT)  PT08.S3(NOx)      NO2(GT)  \
Number Missing      366.000000  1639.000000    366.000000  1642.000000   
Percent Missing       0.039115     0.175163      0.039115     0.175484   

                 PT08.S4(NO2)  PT08.S5(O3)           T          RH          AH  
Number Missing     366.000000   366.000000  366.000000  366.000000  366.000000  
Percent Missing      0.039115     0.039115    0.039115    0.039115    0.039115  
The data is probably not Missing Completely at Random (MCAR).


{'chi_square_stat': 5478.1911, 'df': 88, 'p_value': 0.0}

In [6]:
# Zachycení výstupu MCAR testu
mcar_buffer = io.StringIO()
with contextlib.redirect_stdout(mcar_buffer):
    xg_test.mcar()

# Rozdělení na řádky
mcar_lines = mcar_buffer.getvalue().splitlines()

# Uložení do JSON souboru
with open("../missing/export_mechanisms/mcar_result.json", "w", encoding="utf-8") as f:
    json.dump({"mcar_output": mcar_lines}, f, indent=4, ensure_ascii=False)

In [7]:
xg_test.missing_type()

----- MCAR Test Results -----
Chi Square: 5478.1911
Degrees of Freedom: 88
P-value: 0.0
Conclusion: Reject MCAR at alpha=0.05 (p=0.000e+00).
 Little's MCAR test may be unreliable.
Comment: None

----- MAR vs. MNAR Tests (Feature-wise) -----
      CO(GT) | LRT =  142.718 | P-value = 0.000e+00 | Conclusion: Likely MNAR (reject MAR).
 PT08.S1(CO) | LRT =   -0.906 | P-value = 1.000e+00 | Conclusion: No strong evidence against MAR.
    NMHC(GT) | LRT = 3814.649 | P-value = 0.000e+00 | Conclusion: Likely MNAR (reject MAR).
    C6H6(GT) | LRT =   -0.003 | P-value = 1.000e+00 | Conclusion: No strong evidence against MAR.
PT08.S2(NMHC) | LRT =   -1.544 | P-value = 1.000e+00 | Conclusion: No strong evidence against MAR.
     NOx(GT) | LRT =   32.416 | P-value = 1.244e-08 | Conclusion: Likely MNAR (reject MAR).
PT08.S3(NOx) | LRT =   -0.534 | P-value = 1.000e+00 | Conclusion: No strong evidence against MAR.
     NO2(GT) | LRT =  312.877 | P-value = 0.000e+00 | Conclusion: Likely MNAR (reject MAR)

In [8]:
missing_buffer = io.StringIO()
with contextlib.redirect_stdout(missing_buffer):
    xg_test.missing_type()

missing_lines = missing_buffer.getvalue().splitlines()

with open("../missing/export_mechanisms/missing_type_result.json", "w", encoding="utf-8") as f:
    json.dump({"missing_type_output": missing_lines}, f, indent=4, ensure_ascii=False)